In [1]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt

import sys
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

# Training Data Preparation and Caching

The model uses cached per-chromosome data to help speed up training. Here, we will go through the process of building the necessary tensors and reference files to train the gene prediction model.

## Set up the TrainingDataFormatter

The training data formatter ensures that the correct data files and output directories exist and helps to keep everything standardized.

In [2]:
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")

cell_type = "mESC"
sample_name = "E7.5_rep1"
experiment_name = f"{cell_type}_{sample_name}_tutorial"
organism_code = "mm10"

# Path to the training output directory. Used to store the preprocessing config
output_dir = DATA_DIR / "EXPERIMENTS" / experiment_name

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 19)]

processed_data_dir = DATA_DIR / "PROCESSED_DATA" / experiment_name

# Create the training data formatter object
tdf = data_formatter.TrainingDataFormatter(
    project_dir=PROJECT_DIR,
    experiment_name=experiment_name,
    organism_code=organism_code,
    sample_names=[sample_name],
    processed_data_dir=processed_data_dir,
    chrom_list=chrom_list,
    output_dir=output_dir,
)

Creating output directory: /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/mESC_E7.5_rep1_tutorial


## Sample-Level Data Preparation

We first load the pseudobulk data from the Muon QC and preprocessing pipeline. The data files are loaded and the peak and gene names are standardized.

We load the pseudobulk data for the sample, which also creates a list of gene names. The `split_genes_into_tfs_and_tgs` function checks which of the genes in the gene name list overlap with an organism-specific [file of TF names from CIS-BP](https://mitra.stanford.edu/kundaje/marinovg/oak/various/motifs/CIS-BP/TF_Information_all_motifs.txt) to create `tfs` and `tgs` attributes.


In [ ]:
tg_pseudobulk_path = tdf.file_paths["processed"]["base_dir"] / sample_name / "TG_pseudobulk.parquet"
logging.info(f"TG pseudobulk path: {tg_pseudobulk_path}")
base_pseudobulk_rna_df = pd.read_parquet(tg_pseudobulk_path)


logging.info(f"Base Num Genes: {base_pseudobulk_rna_df.shape[0]}")
logging.info(f"Base Num Metacells: {base_pseudobulk_rna_df.shape[1]}")

In [ ]:
pseudobulk_rna_df = tdf.load_pseudobulk_rna_df(sample_name)

logging.info(f"Number of genes: {len(tdf.genes):,}")

# Next the genes are classified as TFs or TGs by checking which genes in the data are in the reference TF list. 
tdf.tfs, tdf.tgs = tdf.split_genes_into_tfs_and_tgs(tdf.genes)
logging.info(f"  - TFs: {len(tdf.tfs):,} (First 3: {tdf.tfs[:3]})")
logging.info(f"  - TGs: {len(tdf.tgs):,} (First 3: {tdf.tgs[:3]})")

The pseudobulk ATAC data is loaded, which also creates a list of peak names

In [ ]:
pseudobulk_atac_df = tdf.load_pseudobulk_atac_df(sample_name)

logging.info(f"Number of peaks: {len(tdf.peaks):,} (First 3: {tdf.peaks[:3]})")

A BED file of the peak locations is created using `create_peak_bed_file` and the peak locations are stored as a dataframe in `tdf.peak_locs_df`.

In [ ]:
tdf.create_peak_bed_file(pseudobulk_atac_df, sample_name)
display(tdf.peak_locs_df.head())

### Calculate peak to gene distance

Next, we will use the peak bed file created when loading the peak pseudobulk dataset to calculate the distance between each peak and each gene within 1 Mb. This will create a bias tensor to help the model prioritize genes that are closer to peaks when training the peak-TG cross-attention portion of the model.

In [ ]:
tdf.create_peak_to_tg_distance_file(
    sample_name=sample_name,
    force_recalculate=False,
    filter_to_nearest_gene=False,
)

## Chromosome-Specific Data Formatting


In [ ]:
total_TG_pseudobulk_global, pseudobulk_chrom_dict = tdf.aggregate_pseudobulk_datasets(force_recalculate=False)
total_TG_pseudobulk_global.head()

In [ ]:
tdf.create_chrom_files(total_TG_pseudobulk_global, pseudobulk_chrom_dict, force_recalculate=False, verbose=False)

We can check that all of the required cached files exist for the experiment.

In [ ]:
num_missing_files, missing_file_dict = tdf.check_cached_file_exist()
logging.info(f"Missing {num_missing_files} cached files")
missing_file_dict

In [ ]:
tdf.save_settings()